In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / "dataset"
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError("Run the notebook from the project root or its notebooks folder.")

print("Dataset path:", DATASET_ROOT)
print("Folder exists:", DATASET_ROOT.is_dir())

In [ ]:
for split_name in ["train", "test", "unclean"]:
    split_path = DATASET_ROOT / split_name

    print(f"{split_name}: exists={split_path.is_dir()}")

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_counts = []

for split_name in ["train", "test", "unclean"]:
    split_path = DATASET_ROOT / split_name
    total_images = 0

    print(f"\nSplit: {split_name}")

    for class_path in sorted(split_path.iterdir()):
        if not class_path.is_dir():
            continue

        image_paths = [
            path
            for path in class_path.iterdir()
            if path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        ]

        count = len(image_paths)
        total_images += count

        class_counts.append({
            "split": split_name,
            "class_name": class_path.name,
            "count": count,
        })

        print(f"  {class_path.name}: {count}")

    print(f"Total: {total_images}")

In [ ]:
from PIL import Image

sample_path = next(
    path
    for path in sorted((DATASET_ROOT / "train" / "ambulance").iterdir())
    if path.is_file()
    and path.suffix.lower() in IMAGE_EXTENSIONS
)

with Image.open(sample_path) as image:
    image.load()

    print("File:", sample_path.name)
    print("Format:", image.format)
    print("Size:", image.size)
    print("Color mode:", image.mode)

In [ ]:
def inspect_image(image_path):
    result = {
        "path": str(image_path),
        "width": None,
        "height": None,
        "format": None,
        "mode": None,
        "readable": False,
        "error": None,
    }

    try:
        with Image.open(image_path) as image:
            image.load()

            result["width"] = image.width
            result["height"] = image.height
            result["format"] = image.format
            result["mode"] = image.mode
            result["readable"] = True

    except (OSError, ValueError) as error:
        result["error"] = str(error)

    return result

In [ ]:
sample_info = inspect_image(sample_path)
print(sample_info)

In [ ]:
image_records = []

for split_name in ["train", "test", "unclean"]:
    split_path = DATASET_ROOT / split_name

    for class_path in sorted(split_path.iterdir()):
        if not class_path.is_dir():
            continue

        for image_path in sorted(class_path.iterdir()):
            if not image_path.is_file():
                continue

            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            record = inspect_image(image_path)

            record["source_split"] = split_name
            record["label"] = class_path.name

            image_records.append(record)

print("Inspected images:", len(image_records))

unreadable_records = [
    record
    for record in image_records
    if not record["readable"]
]

print("Unreadable images:", len(unreadable_records))

In [ ]:
readable_records = [
    record
    for record in image_records
    if record["readable"]
]

for record in readable_records:
    record["short_side"] = min(record["width"], record["height"])
    record["aspect_ratio"] = record["width"] / record["height"]

for split_name in ["train", "test", "unclean"]:
    split_records = [
        record
        for record in readable_records
        if record["source_split"] == split_name
    ]

    if not split_records:
        continue

    widths = [record["width"] for record in split_records]
    heights = [record["height"] for record in split_records]
    ratios = [record["aspect_ratio"] for record in split_records]

    print(f"\nSplit: {split_name}")
    print(f"Width range: {min(widths)} to {max(widths)}")
    print(f"Height range: {min(heights)} to {max(heights)}")
    print(f"Aspect ratio range: {min(ratios):.2f} to {max(ratios):.2f}")

In [ ]:
from IPython.display import display

train_records = [
    record
    for record in readable_records
    if record["source_split"] == "train"
]

smallest_records = sorted(
    train_records,
    key=lambda record: record["short_side"],
)[:6]

smallest_paths = {
    record["path"]
    for record in smallest_records
}

remaining_records = [
    record
    for record in train_records
    if record["path"] not in smallest_paths
]

most_elongated_records = sorted(
    remaining_records,
    key=lambda record: max(
        record["aspect_ratio"],
        1 / record["aspect_ratio"],
    ),
    reverse=True,
)[:6]

for title, records in [
    ("Smallest images", smallest_records),
    ("Most elongated images", most_elongated_records),
]:
    print(f"\n{title}")

    for record in records:
        print(
            f"Class: {record['label']} | "
            f"Size: {record['width']}x{record['height']} | "
            f"Ratio: {record['aspect_ratio']:.2f}"
        )
        print("Path:", record["path"])

        with Image.open(record["path"]) as image:
            preview = image.convert("RGB")
            preview.thumbnail((320, 320))
            display(preview)

In [ ]:
import hashlib

def compute_image_hash(image_path):
    with Image.open(image_path) as image:
        rgb_image = image.convert("RGB")

        hasher = hashlib.sha256()
        hasher.update(str(rgb_image.size).encode("utf-8"))
        hasher.update(rgb_image.tobytes())

        return hasher.hexdigest()

In [ ]:
sample_hash = compute_image_hash(sample_path)
print(sample_hash)

In [ ]:
hash_groups = {}

for record in readable_records:
    image_hash = compute_image_hash(record["path"])
    record["content_hash"] = image_hash

    if image_hash not in hash_groups:
        hash_groups[image_hash] = []

    hash_groups[image_hash].append(record)

duplicate_groups = [
    group
    for group in hash_groups.values()
    if len(group) > 1
]

print("Images checked:", len(readable_records))
print("Unique image contents:", len(hash_groups))
print("Duplicate groups:", len(duplicate_groups))


extra_copies = sum(
    len(group) - 1
    for group in duplicate_groups
)

print("Extra duplicate copies:", extra_copies)

In [ ]:
cross_split_groups = []
label_conflict_groups = []

for group_number, group in enumerate(duplicate_groups, start=1):
    splits = {record["source_split"] for record in group}
    labels = {record["label"] for record in group}

    if len(splits) > 1:
        cross_split_groups.append(group)

    if len(labels) > 1:
        label_conflict_groups.append(group)

    print(f"\nGroup {group_number}")

    for record in group:
        print(
            f"  {record['source_split']}/{record['label']}"
            f" ﷿﷿﷿ {record['path']}"
        )

    print("Across splits:", len(splits) > 1)
    print("Label conflict:", len(labels) > 1)

print("\nDuplicate groups across splits:", len(cross_split_groups))
print("Groups with conflicting labels:", len(label_conflict_groups))

In [ ]:
for group in label_conflict_groups:
    print("Same image, different labels:")

    for record in group:
        print(
            f"  split={record['source_split']}, "
            f"label={record['label']}, "
            f"path={record['path']}"
        )

    with Image.open(group[0]["path"]) as image:
        preview = image.convert("RGB")
        preview.thumbnail((400, 400))
        display(preview)

In [ ]:
train_hashes = {
    record["content_hash"]
    for record in readable_records
    if record["source_split"] == "train"
}

test_hashes = {
    record["content_hash"]
    for record in readable_records
    if record["source_split"] == "test"
}

known_labels = {
    record["label"]
    for record in readable_records
    if record["source_split"] == "train"
}

seen_hashes = train_hashes | test_hashes

unclean_known_candidates = [
    record
    for record in readable_records
    if record["source_split"] == "unclean"
    and record["label"] in known_labels
    and record["content_hash"] not in seen_hashes
]

neysan_candidates = [
    record
    for record in readable_records
    if record["source_split"] == "unclean"
    and record["label"] == "neysan"
    and record["content_hash"] not in seen_hashes
]

print("New known-class candidates:", len(unclean_known_candidates))
print("New neysan candidates:", len(neysan_candidates))

In [ ]:
from collections import Counter

candidate_counts = Counter(
    record["label"]
    for record in unclean_known_candidates
)

for class_name in sorted(known_labels):
    print(
        f"{class_name:10s} | "
        f"original: 50 | "
        f"new candidates: {candidate_counts[class_name]}"
    )

print("Total new known-class candidates:", sum(candidate_counts.values()))

In [ ]:
import random

SPLIT_SEED = 42
random_generator = random.Random(SPLIT_SEED)

base_records = [
    record
    for record in readable_records
    if record["source_split"] == "train"
]

training_records = []
validation_records = []

for class_name in sorted(known_labels):
    class_records = [
        record
        for record in base_records
        if record["label"] == class_name
    ]

    class_records.sort(key=lambda record: record["path"])
    random_generator.shuffle(class_records)

    validation_records.extend(class_records[:10])
    training_records.extend(class_records[10:])

    print(
        f"{class_name}: "
        f"train={len(class_records[10:])}, "
        f"validation={len(class_records[:10])}"
    )

print("Total train:", len(training_records))
print("Total validation:", len(validation_records))

In [ ]:
training_hashes = {
    record["content_hash"]
    for record in training_records
}

validation_hashes = {
    record["content_hash"]
    for record in validation_records
}

validation_paths = {
    record["path"]
    for record in validation_records
}

validation_indices = [
    index
    for index, record in enumerate(base_records)
    if record["path"] in validation_paths
]

print("Training / validation overlap:",
      len(training_hashes & validation_hashes))

print("Training / test overlap:",
      len(training_hashes & test_hashes))

print("Validation / test overlap:",
      len(validation_hashes & test_hashes))

print("Recorded validation indices:", len(validation_indices))
print("First 10 indices:", validation_indices[:10])

In [ ]:
import json

split_file = PROJECT_ROOT / "reports" / "base_split.json"
split_file.parent.mkdir(parents=True, exist_ok=True)

def relative_paths(records):
    return sorted(
        str(Path(record["path"]).relative_to(PROJECT_ROOT))
        for record in records
    )

split_manifest = {
    "seed": SPLIT_SEED,
    "source": "dataset/train",
    "training_paths": relative_paths(training_records),
    "validation_paths": relative_paths(validation_records),
    "validation_indices": validation_indices,
}

split_file.write_text(
    json.dumps(split_manifest, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("Saved to:", split_file)
print("Training paths:", len(split_manifest["training_paths"]))
print("Validation paths:", len(split_manifest["validation_paths"]))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

review_folders = [
    "train/vanet",
    "unclean/vanet",
    "unclean/neysan",
]

IMAGES_PER_PAGE = 25

for relative_folder in review_folders:
    image_paths = sorted(
        (DATASET_ROOT / relative_folder).glob("*.jpg")
    )
    print(f"{relative_folder}: {len(image_paths)} images")

    for start in range(0, len(image_paths), IMAGES_PER_PAGE):
        page_paths = image_paths[start:start + IMAGES_PER_PAGE]
        fig, axes = plt.subplots(5, 5, figsize=(16, 17))
        axes = axes.ravel()

        for offset, image_path in enumerate(page_paths):
            ax = axes[offset]
            with Image.open(image_path) as image:
                ax.imshow(image.convert("RGB"))
            ax.set_title(
                f"{start + offset + 1}: {image_path.name}",
                fontsize=10,
            )

        for ax in axes:
            ax.axis("off")

        fig.suptitle(
            f"{relative_folder} | "
            f"{start + 1}-{start + len(page_paths)} / {len(image_paths)}"
        )
        plt.tight_layout(rect=(0, 0, 1, 0.97))
        plt.show()
        plt.close(fig)


## بررسی بصری کامل vanet و neysan

برای بررسی رابطهٔ برچسب‌ها، تمام ۵۰ تصویرِ هر یک از پوشه‌های train/vanet، unclean/vanet و unclean/neysan در مجموع ۱۵۰ تصویر، به‌صورت صفحات تصویری مرور شدند. این مرور روی تصاویر اصلیِ پوشه‌ها، پیش از فیلتر تکراری‌ها انجام شده است؛ تعداد ۵۰ در اینجا به معنی ۵۰ نمونهٔ مستقل و مجاز برای افزودن به آموزش نیست. خروجی سلول بالا همهٔ فایل‌ها را با نام و شماره نمایش می‌دهد؛ هیچ نمونه‌گیری تصادفی انجام نمی‌شود.

مشاهدهٔ اصلی: تنوع مدل‌های وانت فقط در unclean/vanet وجود ندارد؛ train/vanet هم وانت‌های مختلف، از جمله خودروهای با ظاهر مشابه نیسان/زامیاد موجود در unclean/neysan، را شامل می‌شود. برای نمونه، train/vanet/194840733.jpg و train/vanet/217970196.jpg از نظر ظاهر با خودروهای متعددی در پوشهٔ neysan مشابه‌اند. این شباهت بصری، تعریف احتمالیِ vanet به‌عنوان یک دستهٔ گسترده و neysan به‌عنوان زیرنوع آن را مطرح می‌کند؛ اما نام پوشه‌ها و تصاویر به‌تنهایی تعریف رسمیِ مرز کلاس‌ها را مشخص نمی‌کنند.

بنابراین neysan برچسبی خارج از هشت برچسب مدل است، ولی لزوماً نوعی خودرو با ویژگی‌های کاملاً ندیده در آموزش نیست. نباید انتظار داشته باشیم مدل صرفاً به‌علت نام neysan، همهٔ این تصاویر را ناشناخته تشخیص دهد یا confidence پایین بدهد. unknown-label و unknown-appearance یک مفهوم نیستند. تحلیل needs_review و confidence باید این هم‌پوشانی معنایی را در نظر بگیرد.

برخی تصاویر شبانه یا دارای پوشیدگی، برای تعیین دقیق مدل خودرو از نمای کوچک قابل‌اتکا نیستند. مرور همهٔ صفحات، به معنی تأیید قطعیِ برچسب تک‌تک تصاویر نیست؛ موارد مبهم به بررسی اندازهٔ اصلی و در صورت لزوم روشن‌سازی تعریف کلاس توسط استاد نیاز دارند.

تصمیم فعلی: نگاشت هشت کلاسِ baseline و فایل‌های اصلی ثابت می‌مانند. داده‌های unclean هنوز به آموزش اضافه نشده‌اند و هیچ فایل یا برچسبی حذف یا تغییر نکرده است. قبل از افزودن نمونه‌ها باید تکراری‌ها و تعارض‌های ثبت‌شده حذفِ منطقی شوند و نمونه‌های مبهم تعیین تکلیف شوند. تکرار محتوایی با دو برچسب متفاوت، شواهد قوی‌تری از تعارض نسبت به صرفاً متفاوت‌بودن ظاهر چند وانت است. neysan برای تحلیل جداگانهٔ رابطهٔ کلاس‌ها و نیاز به بازبینی نگه داشته می‌شود؛ ادغام یا تبدیل آن به کلاس نهم بدون توجیه انجام نمی‌شود.
